<a href="https://colab.research.google.com/github/ParkerC12345/ds2002-fa26/blob/main/notebooks/01-foundations/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [6]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [11]:
# TODO
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print('total revenue:', total_revenue)
print('total units:', total_units)


total revenue: 8520.0
total units: 783


I calculated the revenue and then used the .sum function to find the total for both revenue and units.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [12]:
# TODO
category_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=False)

category_table = category_revenue.to_frame()
category_table['share_percent'] = (category_table['revenue'] / total_revenue * 100).round(1)

print(category_table)

          revenue  share_percent
category                        
Food       4293.0           50.4
Merch      1771.5           20.8
Drink      1554.0           18.2
RainGear    901.5           10.6


I sorted the df by category revuenue using the .sum anbd sort ascending functions. I then made a new table to display this info, and calculated the share percent for each category in it.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [21]:
# TODO
avg_revenue = (df.groupby('vendor_id')['revenue'].mean()).round(1)

order_count = df.groupby('vendor_id')['revenue'].count()

vendor_table = pd.DataFrame({'avg_revenue': avg_revenue, 'order_count': order_count})

vendor_table = vendor_table.sort_values('avg_revenue', ascending=False)

print(vendor_table)

           avg_revenue  order_count
vendor_id                          
V-01              22.6           94
V-18              21.8          108
V-05              20.6           93
V-10              20.3          105


I first calculated the avgerage revenue and the order count to then create a table displaying those respective variables for eahc vendor_id. I used ascending=False to order the vendors by greatest revenue, but the order count helps provide context.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [20]:
# TODO
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_percent = (merch_revenue / total_revenue * 100).round(1)

print(merch_percent)

20.8


I first calcualted the amount of revenue makes to then divide it by the toal_revenue from all categories to merch as a percent of total revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [36]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

df_merged = df.merge(
    vendor_names, on = 'vendor_id',
    how = 'left',
    validate = 'many_to_one'
)

print('rows before:', len(df))
print('rows after:', len(df_merged))

print('revenue before:', df['revenue'].sum())
print('revenue_after:', df_merged['revenue'].sum())

unmatched_vendor = df_merged[df_merged['vendor_name'].isna()]['vendor_id'].unique()
print('unmatched vendor:', unmatched_vendor)


df_merged.loc[df_merged['vendor_id'] == 'V-18', 'vendor_name'] = 'Unknown Vendor'
print(df_merged)

rows before: 400
rows after: 400
revenue before: 8520.0
revenue_after: 8520.0
unmatched vendor: ['V-18']
    vendor_id  category  qty  price  revenue      vendor_name
0        V-10     Drink    2   24.0     48.0  Cav Merch North
1        V-18  RainGear    1   12.0     12.0   Unknown Vendor
2        V-18     Drink    3    4.5     13.5   Unknown Vendor
3        V-10      Food    2   12.0     24.0  Cav Merch North
4        V-18     Drink    3    7.5     22.5   Unknown Vendor
..        ...       ...  ...    ...      ...              ...
395      V-18     Merch    1   12.0     12.0   Unknown Vendor
396      V-01     Merch    2   24.0     48.0     Hoos Burgers
397      V-10      Food    3    7.5     22.5  Cav Merch North
398      V-18     Merch    2   24.0     48.0   Unknown Vendor
399      V-10     Drink    1    7.5      7.5  Cav Merch North

[400 rows x 6 columns]


I first used a left merge to mantain all of the vendors fromt eh orginal df with coressponding vendor names from the new df. I then printed the rows and revenue from before and after the merge to make sure that nothing changed. After this, I spotted that vendor V-18 had 'NaN in place of its vendor name, so I replaced it with 'Unknown Vendor".

**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
# TODO


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
# assert len(df) == 400
# assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
# assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
# assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_your answer here_